# Beat Saber ML Generator - Training Notebook

This notebook trains the Beat Saber level generator on Google Colab.

**Prerequisites:**
1. Upload your `data/processed/` folder to Google Drive
2. Upload the `src/` folder to Google Drive

**Recommended:** Colab Pro for faster GPU and longer runtime

## 1. Setup

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Install dependencies
!pip install torch torchvision torchaudio --quiet
!pip install librosa --quiet

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set paths - UPDATE THESE TO MATCH YOUR DRIVE STRUCTURE
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/beatsaber-ml"

# Verify paths exist
import os
print(f"Project dir exists: {os.path.exists(DRIVE_PROJECT_DIR)}")
print(f"Data dir exists: {os.path.exists(f'{DRIVE_PROJECT_DIR}/data/processed')}")
print(f"Src dir exists: {os.path.exists(f'{DRIVE_PROJECT_DIR}/src')}")

In [ ]:
# Copy data to local storage for faster access
!mkdir -p /content/beatsaber-ml
!cp -r "{DRIVE_PROJECT_DIR}/src" /content/beatsaber-ml/
!cp -r "{DRIVE_PROJECT_DIR}/data/processed" /content/beatsaber-ml/data/

# Create checkpoint dir (will save to Drive)
!mkdir -p "{DRIVE_PROJECT_DIR}/checkpoints"
!mkdir -p "{DRIVE_PROJECT_DIR}/logs"

# Change to project directory
os.chdir('/content/beatsaber-ml')
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Add src to Python path
import sys
sys.path.insert(0, '/content/beatsaber-ml')

# Test imports
from src.data.dataset import BeatSaberDataset, create_data_loaders
from src.data.tokenizer import EventTokenizer
from src.models.generator import BeatSaberGenerator, BeatSaberGeneratorSmall
print("Imports successful!")

## 2. Check Data

In [ ]:
# Load dataset and check stats
dataset = BeatSaberDataset('data/processed')
print(f"Total samples: {len(dataset)}")

stats = dataset.get_stats()
for key, value in stats.items():
    print(f"  {key}: {value}")

In [ ]:
# Create data loaders
BATCH_SIZE = 4  # Adjust based on GPU memory
MAX_AUDIO_LEN = 6000  # Limit sequence length to fit in memory
MAX_TOKEN_LEN = 3000

train_loader, val_loader = create_data_loaders(
    'data/processed',
    batch_size=BATCH_SIZE,
    max_audio_len=MAX_AUDIO_LEN,
    max_token_len=MAX_TOKEN_LEN
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

## 3. Create Model

In [ ]:
import torch

# Settings
MODEL_SIZE = "small"  # Options: "small" (8M), "base" (45M), "large" (76M)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Device: {DEVICE}")
print(f"Model size: {MODEL_SIZE}")

In [ ]:
# Create tokenizer and model
tokenizer = EventTokenizer()
print(f"Vocabulary size: {tokenizer.vocab_size}")

# Get audio feature dimension
sample = dataset[0]
d_audio = sample['audio_features'].size(-1)
print(f"Audio feature dim: {d_audio}")

# Create model
if MODEL_SIZE == "small":
    model = BeatSaberGeneratorSmall(vocab_size=tokenizer.vocab_size, d_audio=d_audio)
else:
    model = BeatSaberGenerator(vocab_size=tokenizer.vocab_size, d_audio=d_audio)

model = model.to(DEVICE)
print(f"Model parameters: {model.count_parameters() / 1e6:.1f}M")

## 4. Training Setup

In [ ]:
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast

# Hyperparameters
EPOCHS = 50
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
WARMUP_EPOCHS = 5

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Learning rate scheduler with warmup
total_steps = len(train_loader) * EPOCHS
warmup_steps = len(train_loader) * WARMUP_EPOCHS

def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    progress = (step - warmup_steps) / (total_steps - warmup_steps)
    return 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159)).item())

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision
scaler = GradScaler()

print(f"Total training steps: {total_steps}")
print(f"Warmup steps: {warmup_steps}")

In [ ]:
# Checkpoint paths (save to Drive for persistence)
CHECKPOINT_DIR = f"{DRIVE_PROJECT_DIR}/checkpoints"
LOG_DIR = f"{DRIVE_PROJECT_DIR}/logs"

print(f"Checkpoints: {CHECKPOINT_DIR}")
print(f"Logs: {LOG_DIR}")

## 5. Training Loop

In [ ]:
import time
import json
from tqdm.notebook import tqdm

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'learning_rate': []
}

best_val_loss = float('inf')
global_step = 0

def train_epoch(model, loader, optimizer, scheduler, scaler, device):
    global global_step
    model.train()
    total_loss = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch in pbar:
        # Move to device
        audio = batch['audio_features'].to(device)
        tokens = batch['token_ids'].to(device)
        difficulty = batch['difficulty'].to(device)
        audio_mask = batch['audio_mask'].to(device)
        token_mask = batch['token_mask'].to(device)
        
        # Teacher forcing
        input_ids = tokens[:, :-1]
        target_ids = tokens[:, 1:]
        input_mask = token_mask[:, :-1]
        
        # Forward pass with AMP
        optimizer.zero_grad()
        
        with autocast():
            logits = model(audio, input_ids, difficulty, 
                          audio_mask=audio_mask, token_mask=input_mask)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                target_ids.reshape(-1),
                ignore_index=0  # PAD token
            )
        
        # Backward pass
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        total_loss += loss.item()
        global_step += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Validating"):
        audio = batch['audio_features'].to(device)
        tokens = batch['token_ids'].to(device)
        difficulty = batch['difficulty'].to(device)
        audio_mask = batch['audio_mask'].to(device)
        token_mask = batch['token_mask'].to(device)
        
        input_ids = tokens[:, :-1]
        target_ids = tokens[:, 1:]
        input_mask = token_mask[:, :-1]
        
        logits = model(audio, input_ids, difficulty,
                      audio_mask=audio_mask, token_mask=input_mask)
        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            target_ids.reshape(-1),
            ignore_index=0
        )
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_loss': val_loss,
        'history': history
    }
    
    torch.save(checkpoint, f"{CHECKPOINT_DIR}/checkpoint_epoch_{epoch}.pt")
    
    if is_best:
        torch.save(checkpoint, f"{CHECKPOINT_DIR}/best_model.pt")
        print(f"  Saved best model!")

In [ ]:
# Main training loop
print("="*60)
print("STARTING TRAINING")
print("="*60)

for epoch in range(EPOCHS):
    epoch_start = time.time()
    
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, DEVICE)
    
    # Validate
    val_loss = validate(model, val_loader, DEVICE)
    
    # Track history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['learning_rate'].append(optimizer.param_groups[0]['lr'])
    
    # Log
    epoch_time = time.time() - epoch_start
    print(f"\nEpoch {epoch+1}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Time: {epoch_time:.1f}s")
    
    # Save checkpoint
    is_best = val_loss < best_val_loss
    if is_best:
        best_val_loss = val_loss
    
    if (epoch + 1) % 5 == 0 or is_best:
        save_checkpoint(model, optimizer, scheduler, epoch+1, val_loss, is_best)

# Save final model
save_checkpoint(model, optimizer, scheduler, EPOCHS, val_loss)

# Save history
with open(f"{LOG_DIR}/training_history.json", 'w') as f:
    json.dump(history, f, indent=2)

print("\n" + "="*60)
print(f"TRAINING COMPLETE! Best val loss: {best_val_loss:.4f}")
print("="*60)

## 6. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Learning rate
axes[1].plot(history['learning_rate'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f"{LOG_DIR}/training_curves.png", dpi=150)
plt.show()

## 7. Test Generation

In [ ]:
# Load best model
checkpoint = torch.load(f"{CHECKPOINT_DIR}/best_model.pt")
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Loaded best model from epoch {checkpoint['epoch']}")
print(f"Validation loss: {checkpoint['val_loss']:.4f}")

In [ ]:
# Generate from a validation sample
sample = val_loader.dataset[0]
if hasattr(sample, 'keys'):
    audio = sample['audio_features']
    difficulty = sample['difficulty']
else:
    audio = sample['audio_features']
    difficulty = sample['difficulty']

# Generate
with torch.no_grad():
    generated = model.generate(
        audio[:1000].unsqueeze(0).to(DEVICE),  # First 1000 ticks
        torch.tensor([difficulty]).to(DEVICE),
        max_len=200,
        temperature=0.8,
        top_k=50
    )

# Decode tokens
generated_tokens = tokenizer.tokens_to_string(generated[0].cpu().tolist())
print(f"Generated {len(generated_tokens)} tokens:")
print(generated_tokens[:30])

## 8. Download Checkpoint

Your trained model is saved in Google Drive at:
- `beatsaber-ml/checkpoints/best_model.pt`

You can download this and use it locally for generation!